# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/medhu07/flyrankai/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!git clone https://github.com/medhu07/flyrankai.git

Cloning into 'flyrankai'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 138 (delta 51), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 1.86 MiB | 11.54 MiB/s, done.
Resolving deltas: 100% (51/51), done.


In [3]:
%cd flyrankai

/content/flyrankai


## 1. My rule and its reason codes

### My baseline rule

I will prioritize pages that have meaningful search visibility but appear to have an opportunity for improvement.

A page receives a higher score when it has:
- relatively high impressions,
- relatively low CTR compared with other visible pages,
- and has not been updated recently.

The score is intentionally simple and transparent. It is a decision-support baseline, not a claim that these pages will definitely improve after a refresh.

### Reason codes

- `visible_low_ctr_stale` — high visibility, relatively low CTR, and old content.
- `visible_low_ctr` — high visibility and relatively low CTR.
- `visible_stale` — high visibility and old content.
- `visible` — high visibility but does not meet the other conditions.

The reason code explains why a page appeared in the ranked queue.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Thresholds are based on the observed starter dataset.
impressions_threshold = df["impressions_90d"].median()
ctr_threshold = df["ctr"].median()
stale_threshold = 180

print("Impressions threshold:", impressions_threshold)
print("CTR threshold:", ctr_threshold)
print("Stale threshold (days):", stale_threshold)

Impressions threshold: 731.0
CTR threshold: 0.07
Stale threshold (days): 180


In [6]:
# Create transparent signals
visible = df["impressions_90d"] >= impressions_threshold
low_ctr = df["ctr"] <= ctr_threshold
stale = df["days_since_last_update"] >= stale_threshold

# Simple baseline score: 1 point for each condition satisfied
df["baseline_score"] = (
    visible.astype(int)
    + low_ctr.astype(int)
    + stale.astype(int)
)

# Assign a reason code
df["reason_code"] = np.select(
    [
        visible & low_ctr & stale,
        visible & low_ctr,
        visible & stale,
        visible
    ],
    [
        "visible_low_ctr_stale",
        "visible_low_ctr",
        "visible_stale",
        "visible"
    ],
    default="other"
)

# Rank pages
df = df.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

df["rank"] = range(1, len(df) + 1)

# Display the top 20
top20 = df[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "impressions_90d",
        "ctr",
        "days_since_last_update"
    ]
].head(20)

display(top20)

,rank,content_id,baseline_score,reason_code,impressions_90d,ctr,days_since_last_update
0,1,content_5feee3994adb,3,visible_low_ctr_stale,7812,0.01,194
1,2,content_b16bd7307b39,3,visible_low_ctr_stale,4590,0.00,194
2,3,content_36ff89c8214e,2,visible_low_ctr,295097,0.05,104
3,4,content_b28d1efd668f,2,visible_low_ctr,286608,0.06,104
4,5,content_8451fc6f034d,2,visible_low_ctr,272144,0.03,20
5,6,content_813e88069237,2,visible_low_ctr,233561,0.06,104
6,7,content_ff94c9b6b411,2,visible_low_ctr,228566,0.04,20
7,8,content_c84a0ab98e90,2,visible_low_ctr,223271,0.03,20
8,9,content_66b4046cc144,2,visible_low_ctr,217415,0.03,20
9,10,content_a023517539fe,2,visible_low_ctr,214047,0.01,20


In [7]:
output_path = Path("work/outputs")
output_path.mkdir(parents=True, exist_ok=True)

output_file = output_path / "baseline_action_score.csv"

df[
    [
        "rank",
        "content_id",
        "client_id",
        "baseline_score",
        "reason_code",
        "impressions_90d",
        "ctr",
        "days_since_last_update"
    ]
].to_csv(output_file, index=False)

print("Saved:", output_file)

Saved: work/outputs/baseline_action_score.csv


## 3. Top-20 Review

I reviewed the top 20 pages produced by the baseline ranking.

The recommended action is to send these pages for human review for a possible content refresh. The reason code explains why each page received its score.

I use moderate confidence because the ranking is based only on a simple rule using historical visibility, CTR, and freshness. A high score does not establish that refreshing the page will improve future performance.

A recommendation could be wrong if low CTR is appropriate for the page's search intent, if the page does not need updating despite being old, or if high impressions do not represent a meaningful refresh opportunity.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create the top-20 review table
top20_review = df.head(20).copy()

top20_review["action"] = "Review for possible refresh"
top20_review["confidence_note"] = (
    "Moderate confidence; rule-based recommendation"
)
top20_review["what_could_make_it_wrong"] = (
    "Search intent or page context may make a refresh unnecessary"
)

display(
    top20_review[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "impressions_90d",
            "ctr",
            "days_since_last_update",
            "action",
            "confidence_note",
            "what_could_make_it_wrong"
        ]
    ]
)


,rank,content_id,baseline_score,reason_code,impressions_90d,ctr,days_since_last_update,action,confidence_note,what_could_make_it_wrong
0,1,content_5feee3994adb,3,visible_low_ctr_stale,7812,0.01,194,Review for possible refresh,Moderate confidence; rule-based recommendation,Search intent or page context may make a refre...
1,2,content_b16bd7307b39,3,visible_low_ctr_stale,4590,0.00,194,Review for possible refresh,Moderate confidence; rule-based recommendation,Search intent or page context may make a refre...
2,3,content_36ff89c8214e,2,visible_low_ctr,295097,0.05,104,Review for possible refresh,Moderate confidence; rule-based recommendation,Search intent or page context may make a refre...
3,4,content_b28d1efd668f,2,visible_low_ctr,286608,0.06,104,Review for possible refresh,Moderate confidence; rule-based recommendation,Search intent or page context may make a refre...
4,5,content_8451fc6f034d,2,visible_low_ctr,272144,0.03,20,Review for possible refresh,Moderate confidence; rule-based recommendation,Search intent or page context may make a refre...
5,6,content_813e88069237,2,visible_low_ctr,233561,0.06,104,Review for possible refresh,Moderate confidence; rule-based recommendation,Search intent or page context may make a refre...
6,7,content_ff94c9b6b411,2,visible_low_ctr,228566,0.04,20,Review for possible refresh,Moderate confidence; rule-based recommendation,Search intent or page context may make a refre...
7,8,content_c84a0ab98e90,2,visible_low_ctr,223271,0.03,20,Review for possible refresh,Moderate confidence; rule-based recommendation,Search intent or page context may make a refre...
8,9,content_66b4046cc144,2,visible_low_ctr,217415,0.03,20,Review for possible refresh,Moderate confidence; rule-based recommendation,Search intent or page context may make a refre...
9,10,content_a023517539fe,2,visible_low_ctr,214047,0.01,20,Review for possible refresh,Moderate confidence; rule-based recommendation,Search intent or page context may make a refre...


## 4. Weak Picks + Leakage Check

The top-20 review shows that the baseline is transparent and easy to interpret, but it also produces some weak picks.

Several highly ranked pages have high impressions and low CTR but were updated recently. For example, some pages in the top 20 were updated only 7, 20, or 25 days ago. These pages may not actually need a content refresh, even though the rule ranks them highly.

This is a limitation of the simple baseline: it treats low CTR as an opportunity without knowing the page's search intent, content quality, or whether the CTR is appropriate for its position.

The baseline uses only historical fields available in the starter snapshot: impressions, CTR, and days since last update. It does not use `trend_direction` or `trend_pct`, which are excluded because they are trend-derived and could leak target information. IDs such as `content_id` and `client_id` are used only for identification and grouping, not scoring.

The ranked queue should therefore be treated as decision-support for human review, not as an automatic instruction to refresh a page.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Fields actually used by the baseline score
baseline_features = [
    "impressions_90d",
    "ctr",
    "days_since_last_update"
]

# Fields explicitly excluded from scoring
excluded_fields = [
    "trend_direction",
    "trend_pct"
]

print("Baseline features:")
for col in baseline_features:
    print("-", col)

print("\nExcluded trend-derived fields:")
for col in excluded_fields:
    print("-", col)

# Confirm the excluded fields are not part of the scoring calculation
score_expression = "visible + low_ctr + stale"

print("\nBaseline score definition:")
print(score_expression)

print("\nExcluded fields are not used in the baseline score.")

Baseline features:
- impressions_90d
- ctr
- days_since_last_update

Excluded trend-derived fields:
- trend_direction
- trend_pct

Baseline score definition:
visible + low_ctr + stale

Excluded fields are not used in the baseline score.


In [10]:
recent_top20 = df[
    (df["rank"] <= 20) &
    (df["days_since_last_update"] < 30)
]

print("Top-20 pages updated within the last 30 days:", len(recent_top20))

display(
    recent_top20[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "impressions_90d",
            "ctr",
            "days_since_last_update"
        ]
    ]
)

Top-20 pages updated within the last 30 days: 9


,rank,content_id,baseline_score,reason_code,impressions_90d,ctr,days_since_last_update
4,5,content_8451fc6f034d,2,visible_low_ctr,272144,0.03,20
6,7,content_ff94c9b6b411,2,visible_low_ctr,228566,0.04,20
7,8,content_c84a0ab98e90,2,visible_low_ctr,223271,0.03,20
8,9,content_66b4046cc144,2,visible_low_ctr,217415,0.03,20
9,10,content_a023517539fe,2,visible_low_ctr,214047,0.01,20
11,12,content_0e70a832cb7a,2,visible_low_ctr,173450,0.04,25
13,14,content_e12868d1f396,2,visible_low_ctr,149712,0.07,7
16,17,content_453722754fea,2,visible_low_ctr,140079,0.01,20
17,18,content_7158cfbbc450,2,visible_low_ctr,134567,0.06,20


### Weak-pick observation

The top-20 review identified 9 pages that had been updated within the last 30 days. These pages received high rankings because they had high impressions and low CTR, despite being recently updated. This suggests that the baseline can over-prioritize visible pages with low CTR and does not fully capture whether a page genuinely needs a refresh.

This is why the score should be treated as a decision-support queue for human review rather than an automatic refresh recommendation.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.